In [1]:
import sounddevice as sd

print("=== AVAILABLE AUDIO INPUTS ===")
for i, d in enumerate(sd.query_devices()):
    # Only show devices that can actually record audio
    if d['max_input_channels'] > 0:
        print(f"ID {i}: {d['name']} (Channels: {d['max_input_channels']})")

=== AVAILABLE AUDIO INPUTS ===
ID 0: Microsoft Sound Mapper - Input (Channels: 2)
ID 1: Microphone (DroidCam Audio) (Channels: 2)
ID 2: CABLE Output (VB-Audio Virtual  (Channels: 16)
ID 3: Microphone Array (Intel® Smart  (Channels: 4)
ID 8: Primary Sound Capture Driver (Channels: 2)
ID 9: Microphone (DroidCam Audio) (Channels: 2)
ID 10: CABLE Output (VB-Audio Virtual Cable) (Channels: 16)
ID 11: Microphone Array (Intel® Smart Sound Technology for Digital Microphones) (Channels: 4)
ID 19: CABLE Output (VB-Audio Virtual Cable) (Channels: 2)
ID 20: Microphone (DroidCam Audio) (Channels: 1)
ID 21: Microphone Array (Intel® Smart Sound Technology for Digital Microphones) (Channels: 4)
ID 22: CABLE Output (VB-Audio Point) (Channels: 16)
ID 24: Input (VB-Audio Point) (Channels: 16)
ID 25: Microphone Array 1 () (Channels: 4)
ID 26: Microphone Array 2 () (Channels: 4)
ID 30: PC Speaker (Realtek HD Audio output with SST) (Channels: 2)
ID 31: Stereo Mix (Realtek HD Audio Stereo input) (Channels: 2

In [2]:
# Topics to detect — edit this list to change what the agent monitors.
# This list is used for both transcript labeling (auto_label_json_files)
# and evaluation (run_planning_eval, run_e2e_eval).
TARGET_TOPICS = [
    "dividend",
    "gross margin",
    "inventory",
    "capital allocation",
    "net income",
    "operating income",
    "pricing",
    "capex",
    "investment portfolio",
    "sales growth",
]

print(f"Topics configured: {len(TARGET_TOPICS)}")
for t in TARGET_TOPICS:
    print(f"  - {t}")

Topics configured: 10
  - dividend
  - gross margin
  - inventory
  - capital allocation
  - net income
  - operating income
  - pricing
  - capex
  - investment portfolio
  - sales growth


In [3]:
from transcript_processing import process_aligned_files, auto_label_json_files

# Convert .aligned.nlp transcript files to 15-second chunked JSON
process_aligned_files('testset_transcript')

Converted: 4462231 -> testset_transcript\4462231_parsed.json
Converted: 4474506 -> testset_transcript\4474506_parsed.json
Converted: 4485192 -> testset_transcript\4485192_parsed.json
Converted: 4485206 -> testset_transcript\4485206_parsed.json
Converted: 4485244 -> testset_transcript\4485244_parsed.json


In [4]:
# Label each chunk: topic_present=1 if it contains any of the target topics
auto_label_json_files('testset_transcript', keywords=TARGET_TOPICS)

Found 5 files in 'testset_transcript'. Starting auto-labeling...

Saved 4462231_labeled.json
   -> Topics detected in 27 out of 166 chunks.
Saved 4474506_labeled.json
   -> Topics detected in 28 out of 265 chunks.
Saved 4485192_labeled.json
   -> Topics detected in 31 out of 247 chunks.
Saved 4485206_labeled.json
   -> Topics detected in 8 out of 155 chunks.
Saved 4485244_labeled.json
   -> Topics detected in 6 out of 246 chunks.

All files labeled successfully.


# Offline Evaluation — Perception · Planning · End-to-End

Three sections, all using the same stratified sample of audio chunks:

1. **Perception (WER)** — Whisper & PocketSphinx word-error rates on real audio
2. **Planning (GT text)** — P/R/F1 for each planner on ground-truth transcripts (upper bound — no transcription errors)
3. **End-to-end** — P/R/F1 for the four production combos (transcribed audio → detection)

Section 2 vs Section 3 on identical chunks gives the pure perception penalty: `gap = Sec2 F1 - Sec3 F1`.

In [5]:
import os, sys, builtins, importlib

PROJECT_DIR = r"C:\Users\beung-yoga\Documents\GitHub4\meeting-alert-agent"
os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

import eval as _eval
importlib.reload(_eval)   # force re-read of eval.py so new functions are visible

# Reset the output log so re-running the notebook starts fresh.
_eval._output_log.clear()

# Patch eval's print helpers for Jupyter:
#   _p        — drop 'end' kwarg (avoids \r overwrite) AND append to _output_log
#               so _save_results() captures the full eval output.
#   _progress — suppress in-place ticker (not useful in a notebook cell).
def _jupyter_p(msg, **kw):
    kw.pop("end", None)
    builtins.print(msg, **kw)
    _eval._output_log.append(str(msg))

_eval._p        = _jupyter_p
_eval._progress = lambda label, done, total, t0: None

from agent_core import LETTERS
from eval import (
    _load_labeled_files, _stratified_sample,
    _save_sample_cache, _load_sample_cache,
    _build_sample, _sample_cache_is_valid,
    _load_whisper_with_progress, _save_results,
    WHISPER_DEVICE, WHISPER_COMPUTE_TYPE, AUDIO_DIR,
    MAX_AUDIO_PER_FILE, MAX_CHUNKS_LLM, SAMPLE_CACHE_PATH,
    PLANNING_MODES, E2E_COMBOS,
    run_perception_eval, run_planning_eval, run_e2e_eval,
)
print("Setup complete")

Setup complete


In [6]:
whisper_model = _load_whisper_with_progress(WHISPER_DEVICE, WHISPER_COMPUTE_TYPE)
print("Whisper ready")

    (torch CUDA pre-init: NVIDIA GeForce RTX 4060 Laptop GPU)
    (model cached — loading from disk…)
Whisper ready


In [7]:
from sentence_transformers import SentenceTransformer
import speech_recognition as sr

semantic_model = SentenceTransformer("all-MiniLM-L6-v2")
recognizer     = sr.Recognizer()
print("SentenceTransformer and PocketSphinx ready")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


SentenceTransformer and PocketSphinx ready


In [8]:
# Build the topic list in the format expected by detect_topics()
eval_topics = [{"id": LETTERS[i], "text": t} for i, t in enumerate(TARGET_TOPICS)]

files_all    = _load_labeled_files()
total_chunks = sum(len(c) for _, c in files_all)
total_pos    = sum(c["topic_present"] for _, cs in files_all for c in cs)

dataset_sample, llm_subset = None, None

if SAMPLE_CACHE_PATH and os.path.exists(SAMPLE_CACHE_PATH):
    _eval._p(f"  Loading sample from cache: {SAMPLE_CACHE_PATH}")
    dataset_sample, llm_subset = _load_sample_cache(files_all, SAMPLE_CACHE_PATH)
    if not _sample_cache_is_valid(dataset_sample, llm_subset):
        llm_pos_rate = sum(c["topic_present"] for c in llm_subset) / max(len(llm_subset), 1)
        _eval._p(f"  Cache invalid (LLM subset positive rate = {llm_pos_rate:.1%}) — regenerating.")
        os.remove(SAMPLE_CACHE_PATH)
        dataset_sample, llm_subset = None, None

if dataset_sample is None:
    dataset_sample, llm_subset = _build_sample(files_all)
    if SAMPLE_CACHE_PATH:
        _save_sample_cache(dataset_sample, llm_subset, SAMPLE_CACHE_PATH)
        _eval._p(f"  Sample saved: {SAMPLE_CACHE_PATH}  (delete to re-sample)")

flat_sample  = [c for _, cs in dataset_sample for c in cs]
sample_total = len(flat_sample)
sample_pos   = sum(c["topic_present"] for c in flat_sample)
llm_total    = len(llm_subset)
llm_pos      = sum(c["topic_present"] for c in llm_subset)

_eval._p(f"  Full corpus   : {len(files_all)} files | {total_chunks} chunks | {total_pos} positive ({100*total_pos/total_chunks:.1f}%)")
_eval._p(f"  Shared sample : {sample_total} chunks | {sample_pos} positive ({100*sample_pos/sample_total:.1f}%) | <={MAX_AUDIO_PER_FILE}/file, all positives preserved")
_eval._p(f"  LLM subset    : {llm_total} chunks | {llm_pos} positive ({100*llm_pos/llm_total:.1f}%) | stratified from shared set")
_eval._p(f"  Topics        : {len(eval_topics)} ({', '.join(t['text'] for t in eval_topics)})")
_eval._p(f"  Sec 2 vs Sec 3 on identical chunks — gap = perception penalty.")

  Sample saved: eval_sample_cache.json  (delete to re-sample)
  Full corpus   : 5 files | 1079 chunks | 100 positive (9.3%)
  Shared sample : 200 chunks | 100 positive (50.0%) | <=40/file, all positives preserved
  LLM subset    : 50 chunks | 25 positive (50.0%) | stratified from shared set
  Topics        : 10 (dividend, gross margin, inventory, capital allocation, net income, operating income, pricing, capex, investment portfolio, sales growth)
  Sec 2 vs Sec 3 on identical chunks — gap = perception penalty.


## Section 1 — Perception: Word Error Rate

Transcribes the shared audio sample with Whisper (small, CUDA) and PocketSphinx, then measures WER against ground-truth transcripts.

In [9]:
transcripts = run_perception_eval(dataset_sample, whisper_model, recognizer)

                                                                                          
          SECTION 1  ·  PERCEPTION — Word Error Rate (WER)          
  Shared evaluation set: 200 chunks across 5 files (100 positive, 50.0%)
  (Stratified per file: all positives kept when under budget, proportional otherwise; max 40/file)

  ✓ 4462231  (40 chunks, 27 positive)          
  ✓ 4474506  (40 chunks, 28 positive)          
  ✓ 4485192  (40 chunks, 31 positive)          
  ✓ 4485206  (40 chunks, 8 positive)          
  ✓ 4485244  (40 chunks, 6 positive)          

  Model                    Avg WER   Chunks
  ──────────────────────  ────────  ───────
  Whisper (small)           0.4697      200
  PocketSphinx              0.8777      200

  Lower WER = better. WER 1.0 means as many errors as reference words.


## Section 2 — Planning: P/R/F1 on Ground-Truth Text

Feeds the real transcript text to each planning model (no audio errors). This is the **upper bound** for detection quality.

In [10]:
planning_results = run_planning_eval(dataset_sample, llm_subset, semantic_model, eval_topics=eval_topics)

                                                                                          
           SECTION 2  ·  PLANNING — on Ground Truth Text           
  (Same chunks as Sec 1/3 — GT text gives the upper bound for planning)

  Shared set : 200 chunks | 100 positive (50.0%)
  LLM subset : 50 chunks | 25 positive (50.0%)  [stratified — same subset used in Sec 3]

  Planning Mode                     Prec     Rec      F1    TP    FP    FN       N
  ──────────────────────────────  ──────  ──────  ──────  ────  ────  ────  ──────
  LLM (Qwen 2.5)                   0.828   0.960   0.889    24     5     1     50*    
  LLM (Gemma 2)                    0.767   0.920   0.836    23     7     2     50*    
  Transformer (Embeddings)         0.919   0.680   0.782    68     6    32    200     
  Non-DL (Keywords)                0.775   1.000   0.873   100    29     0    200     

  * LLM subset: 50 stratified chunks (25 positive, 50.0%) — identical to Sec 3 LLM rows.


## Section 3 — End-to-End: P/R/F1 on Transcribed Audio

Uses the transcriptions from Section 1. Requires Section 1 to have been run first.  
Compare against Section 2 on the same chunks to isolate the perception penalty.

In [11]:
run_e2e_eval(transcripts, llm_subset, semantic_model, eval_topics=eval_topics)

                                                                                          
                SECTION 3  ·  END-TO-END EVALUATION                
  (Same chunks as Sec 1/2 — transcribed text reveals perception penalty)

  Shared set : 200 transcribed chunks (100 positive, 50.0%)
  LLM subset : 50 chunks (25 positive, 50.0%)  [same subset as Sec 2 LLM rows]

  Perception        Planning                          Prec     Rec      F1    TP    FP    FN       N
  ────────────────  ──────────────────────────────  ──────  ──────  ──────  ────  ────  ────  ──────
  Whisper           LLM (Qwen 2.5)                   0.846   0.880   0.863    22     4     3     50*    
  Whisper           LLM (Gemma 2)                    0.647   0.880   0.746    22    12     3     50*    
  Whisper           Transformer (Embeddings)         0.873   0.620   0.725    62     9    38    200     
  PocketSphinx      Non-DL (Keywords)                0.756   0.310   0.440    31    10    69    200     

  * 

In [12]:
result_path = _save_results()
_eval._p(f"  Results saved: {result_path}")

  Results saved: eval_results\eval_20260421_014311.txt
